# CLIP + PANNs + fixed mapping — development 6000

Development-only decision-fusion baseline. PANNs event scores are mapped to the
10 scene classes with the fixed event-to-scene table, normalized, and combined
with CLIP using an alpha sweep. The experiment logic is unchanged from the
working notebook; only code layout and comments have been simplified.

## Setup

In [6]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report
PROJECT_ROOT = Path('C:\\Users\\22313906t\\Desktop\\TAU_project')
CLIP_PATH = PROJECT_ROOT / 'results_CLIP' / 'tau_6000_clip_predictions.csv'
PANNS_PATH = PROJECT_ROOT / 'results_PANNs_6000' / 'tau_6000_panns_events.csv'
FUSION_DIR = PROJECT_ROOT / 'results_FUSION_6000'
FUSION_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = FUSION_DIR / 'tau_6000_clip_panns_naive.csv'
SWEEP_PATH = FUSION_DIR / 'tau_6000_clip_panns_alpha_sweep.csv'
PER_CLASS_PATH = FUSION_DIR / 'tau_6000_clip_panns_per_class.csv'
TAU_CLASSES = [
    'airport',
    'bus',
    'metro',
    'metro station',
    'park',
    'public square',
    'shopping mall',
    'pedestrian street',
    'street traffic',
    'tram',
]
CLIP_SCORE_COLS = {scene: scene.replace(' ', '_') + '_score' for scene in TAU_CLASSES}


def norm_event(x):
    x = str(x).lower()
    x = re.sub('[^a-z0-9]+', ' ', x)
    return re.sub('\\s+', ' ', x).strip()


## Fixed event-to-scene mapping

In [7]:
AUDIO_SCENE_MAP = {
    'bus': {'bus': 1.0},
    'air brake': {'bus': 0.9, 'street traffic': 0.4},
    'vehicle': {'bus': 0.55, 'metro': 0.4, 'metro station': 0.3, 'street traffic': 0.75, 'tram': 0.55},
    'motor vehicle road': {'bus': 0.25, 'street traffic': 1.0, 'tram': 0.15},
    'car': {'bus': 0.2, 'street traffic': 1.0, 'tram': 0.1},
    'engine': {'bus': 0.3, 'street traffic': 0.75, 'tram': 0.15},
    'vehicle horn car horn honking': {'bus': 0.2, 'street traffic': 0.9, 'tram': 0.15},
    'train': {'metro': 1.0, 'metro station': 0.9, 'tram': 0.7},
    'rail transport': {'metro': 1.0, 'metro station': 0.9, 'tram': 0.75},
    'railroad car train wagon': {'metro': 1.0, 'metro station': 0.85, 'tram': 0.7},
    'railroad car': {'metro': 1.0, 'metro station': 0.85, 'tram': 0.7},
    'train wheels squealing': {'metro': 0.85, 'metro station': 1.0, 'tram': 0.65},
    'subway metro underground': {'metro': 1.0, 'metro station': 1.0, 'tram': 0.15},
    'sliding door': {'bus': 0.3, 'metro': 0.45, 'metro station': 0.65, 'tram': 0.45},
    'door': {'bus': 0.15, 'metro': 0.2, 'metro station': 0.35, 'tram': 0.2},
    'bird': {'park': 1.0, 'public square': 0.1, 'pedestrian street': 0.1},
    'bird vocalization bird call bird song': {
        'park': 1.0,
        'public square': 0.1,
        'pedestrian street': 0.1,
    },
    'outside urban or manmade': {
        'public square': 0.8,
        'pedestrian street': 0.8,
        'street traffic': 0.55,
        'park': 0.2,
    },
    'aircraft': {'airport': 1.0},
    'aircraft engine': {'airport': 0.9},
    'jet engine': {'airport': 0.9},
    'fixed wing aircraft airplane': {'airport': 1.0},
}


## Load and prepare predictions

In [8]:
clip_df = pd.read_csv(CLIP_PATH)
clip_df['clip_id'] = clip_df['clip_id'].astype(str)
print('=' * 70)
print('CLIP + PANNs')
print('=' * 70)
print('CLIP clips:', clip_df['clip_id'].nunique())
for scene, col in CLIP_SCORE_COLS.items():
    if col not in clip_df.columns:
        raise ValueError(f'Missing CLIP score column: {col}')
experiment_ids = set(clip_df['clip_id'])
panns_df = pd.read_csv(PANNS_PATH)
panns_df['clip_id'] = panns_df['clip_id'].astype(str)
possible_label_cols = ['label', 'event', 'event_label', 'audio_event', 'class_name']
EVENT_COL = None
for col in possible_label_cols:
    if col in panns_df.columns:
        EVENT_COL = col
        break
if EVENT_COL is None:
    raise ValueError(f'Could not find the PANNs event-label column.\nAvailable columns:\n{list(panns_df.columns)}')
possible_score_cols = ['max_score', 'score', 'mean_score']
SCORE_COL = None
for col in possible_score_cols:
    if col in panns_df.columns:
        SCORE_COL = col
        break
if SCORE_COL is None:
    raise ValueError(f'Could not find PANNs score column.\nAvailable columns:\n{list(panns_df.columns)}')
print('PANNs event column:', EVENT_COL)
print('PANNs score column:', SCORE_COL)
panns_df = panns_df[panns_df['clip_id'].isin(experiment_ids)].copy()
print('PANNs matched clips:', panns_df['clip_id'].nunique())
print('PANNs event rows:', len(panns_df))
panns_df['event_norm'] = panns_df[EVENT_COL].apply(norm_event)
audio_rows = []
for clip_id, group in panns_df.groupby('clip_id'):
    raw_scores = {scene: 0.0 for scene in TAU_CLASSES}
    for _, row in group.iterrows():
        event = row['event_norm']
        try:
            event_score = float(row[SCORE_COL])
        except:
            continue
        if not np.isfinite(event_score):
            continue
        if event not in AUDIO_SCENE_MAP:
            continue
        mapping = AUDIO_SCENE_MAP[event]
        for scene, weight in mapping.items():
            raw_scores[scene] += event_score * weight
    total = sum(raw_scores.values())
    if total > 0:
        audio_scores = {scene: raw_scores[scene] / total for scene in TAU_CLASSES}
        has_audio_evidence = True
    else:
        audio_scores = {scene: 0.0 for scene in TAU_CLASSES}
        has_audio_evidence = False
    row_out = {'clip_id': clip_id, 'has_audio_evidence': has_audio_evidence}
    for scene in TAU_CLASSES:
        row_out['audio_' + scene.replace(' ', '_') + '_score'] = audio_scores[scene]
    audio_rows.append(row_out)
audio_score_df = pd.DataFrame(audio_rows)
fusion_df = clip_df.merge(audio_score_df, on='clip_id', how='left')
audio_score_cols = ['audio_' + scene.replace(' ', '_') + '_score' for scene in TAU_CLASSES]
fusion_df[audio_score_cols] = fusion_df[audio_score_cols].fillna(0.0)
fusion_df['has_audio_evidence'] = fusion_df['has_audio_evidence'].fillna(False)
print(
    '\nClips with mapped PANNs evidence:',
    int(fusion_df['has_audio_evidence'].sum()),
    '/',
    len(fusion_df),
)
clip_accuracy = accuracy_score(fusion_df['ground_truth'], fusion_df['clip_top1'])
clip_macro_f1 = f1_score(
    fusion_df['ground_truth'],
    fusion_df['clip_top1'],
    labels=TAU_CLASSES,
    average='macro',
    zero_division=0,
)
print('\nOriginal CLIP accuracy:', f'{clip_accuracy * 100:.2f}%')
print('Original CLIP Macro-F1:', f'{clip_macro_f1:.4f}')


CLIP + PANNs
CLIP clips: 6000
PANNs event column: label
PANNs score column: max_score
PANNs matched clips: 6000
PANNs event rows: 86665

Clips with mapped PANNs evidence: 5983 / 6000

Original CLIP accuracy: 78.93%
Original CLIP Macro-F1: 0.7877


## Fusion-weight sweep

In [9]:
def evaluate_alpha(df, alpha):
    predictions = []
    top_scores = []
    for _, row in df.iterrows():
        clip_scores = np.array([float(row[CLIP_SCORE_COLS[scene]]) for scene in TAU_CLASSES])
        audio_scores = np.array([float(row['audio_' + scene.replace(' ', '_') + '_score']) for scene in TAU_CLASSES])
        if audio_scores.sum() <= 0:
            fused = clip_scores
        else:
            fused = alpha * clip_scores + (1.0 - alpha) * audio_scores
        best_index = int(np.argmax(fused))
        predictions.append(TAU_CLASSES[best_index])
        top_scores.append(float(fused[best_index]))
    accuracy = accuracy_score(df['ground_truth'], predictions)
    macro_f1 = f1_score(
        df['ground_truth'],
        predictions,
        labels=TAU_CLASSES,
        average='macro',
        zero_division=0,
    )
    return (predictions, top_scores, accuracy, macro_f1)


ALPHAS = [0.95, 0.9, 0.85, 0.8, 0.75, 0.7, 0.65, 0.6]
sweep_rows = []
for alpha in ALPHAS:
    preds, scores, acc, mf1 = evaluate_alpha(fusion_df, alpha)
    sweep_rows.append({'clip_weight_alpha': alpha, 'audio_weight': 1.0 - alpha, 'accuracy': acc, 'accuracy_percent': acc * 100, 'macro_f1': mf1, 'improvement_vs_clip_points': (acc - clip_accuracy) * 100})
sweep_df = pd.DataFrame(sweep_rows)
sweep_df.to_csv(SWEEP_PATH, index=False)
print('\n' + '=' * 70)
print('CLIP + PANNs ALPHA SWEEP')
print('=' * 70)
print(sweep_df.round(4).to_string(index=False))
best_row = sweep_df.sort_values(['accuracy', 'macro_f1'], ascending=False).iloc[0]
BEST_ALPHA = float(best_row['clip_weight_alpha'])



CLIP + PANNs ALPHA SWEEP
 clip_weight_alpha  audio_weight  accuracy  accuracy_percent  macro_f1  improvement_vs_clip_points
              0.95          0.05    0.7895           78.9500    0.7880                      0.0167
              0.90          0.10    0.7892           78.9167    0.7876                     -0.0167
              0.85          0.15    0.7902           79.0167    0.7887                      0.0833
              0.80          0.20    0.7895           78.9500    0.7880                      0.0167
              0.75          0.25    0.7898           78.9833    0.7884                      0.0500
              0.70          0.30    0.7895           78.9500    0.7880                      0.0167
              0.65          0.35    0.7883           78.8333    0.7868                     -0.1000
              0.60          0.40    0.7878           78.7833    0.7862                     -0.1500


## Final evaluation and outputs

In [10]:
best_predictions, best_scores, best_accuracy, best_macro_f1 = evaluate_alpha(fusion_df, BEST_ALPHA)
fusion_df['panns_fused_top1'] = best_predictions
fusion_df['panns_fused_top1_score'] = best_scores
fusion_df['clip_correct'] = fusion_df['clip_top1'] == fusion_df['ground_truth']
fusion_df['fusion_correct'] = fusion_df['panns_fused_top1'] == fusion_df['ground_truth']
fusion_df[
    'change_type'] = np.select([~fusion_df['clip_correct'] & fusion_df['fusion_correct'], fusion_df['clip_correct'] & ~fusion_df['fusion_correct'], fusion_df['clip_correct'] & fusion_df['fusion_correct']],
    ['Wrong->Correct', 'Correct->Wrong', 'Correct->Correct'],
    default='Wrong->Wrong',
)
change_counts = fusion_df['change_type'].value_counts()
per_class_rows = []
for scene in TAU_CLASSES:
    temp = fusion_df[fusion_df['ground_truth'] == scene]
    clip_acc = (temp['clip_top1'] == scene).mean()
    fused_acc = (temp['panns_fused_top1'] == scene).mean()
    per_class_rows.append({'class': scene, 'number_of_clips': len(temp), 'clip_accuracy_percent': clip_acc * 100, 'clip_panns_accuracy_percent': fused_acc * 100, 'change_points': (fused_acc - clip_acc) * 100})
per_class_df = pd.DataFrame(per_class_rows)
per_class_df.to_csv(PER_CLASS_PATH, index=False)
fusion_df.to_csv(OUTPUT_PATH, index=False)
print('\n' + '=' * 70)
print('BEST EXPLORATORY CLIP + PANNs RESULT')
print('=' * 70)
print('Best CLIP weight:', BEST_ALPHA)
print('PANNs weight:', round(1.0 - BEST_ALPHA, 2))
print('\nCLIP:', f'{clip_accuracy * 100:.2f}%')
print('CLIP + PANNs:', f'{best_accuracy * 100:.2f}%')
print('Improvement:', f'{(best_accuracy - clip_accuracy) * 100:+.2f}', 'percentage points')
print('\nCLIP Macro-F1:', f'{clip_macro_f1:.4f}')
print('Fusion Macro-F1:', f'{best_macro_f1:.4f}')
print('\nCorrection analysis:')
for label in ['Wrong->Correct', 'Correct->Wrong', 'Correct->Correct', 'Wrong->Wrong']:
    print(f'{label:18s}:', int(change_counts.get(label, 0)))
print('\nPer-class results:')
print(per_class_df.round(2).to_string(index=False))
print('\nSaved:')
print(OUTPUT_PATH)
print(SWEEP_PATH)
print(PER_CLASS_PATH)
print('\nIMPORTANT:')
print('This alpha sweep is exploratory only.')
print('For the paper, choose alpha on VALIDATION and report performance on untouched TEST data.')



BEST EXPLORATORY CLIP + PANNs RESULT
Best CLIP weight: 0.85
PANNs weight: 0.15

CLIP: 78.93%
CLIP + PANNs: 79.02%
Improvement: +0.08 percentage points

CLIP Macro-F1: 0.7877
Fusion Macro-F1: 0.7887

Correction analysis:
Wrong->Correct    : 16
Correct->Wrong    : 11
Correct->Correct  : 4725
Wrong->Wrong      : 1248

Per-class results:
            class  number_of_clips  clip_accuracy_percent  clip_panns_accuracy_percent  change_points
          airport              600                  80.33                        80.17          -0.17
              bus              600                  54.67                        54.33          -0.33
            metro              600                  96.33                        96.50           0.17
    metro station              600                  74.17                        73.83          -0.33
             park              600                  89.83                        90.67           0.83
    public square              600                 